# 01 - Ingest and validate

Phase 1. Load BPI Challenge 2019 through `FileSource`, apply the
`configs/bpi2019.yaml` mapping preset, and print the `DataQualityReport`.
No cleaning is applied here without an explicit policy.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # PP_BPI2019_PATH, PP_MODEL

from processpilot.ingest.mapping import apply_mapping, load_preset, split_by_segment
from processpilot.ingest.validate import validate
from processpilot.sources.file_source import FileSource

## Fetch the raw log

First run parses the full 695 MB XES (several minutes); subsequent runs read
the size+mtime-keyed parquet cache in `data/processed/`.

In [2]:
source_path = Path(os.environ["PP_BPI2019_PATH"])
source = FileSource(source_path)
raw_df = source.fetch()
source.describe()

C:\Users\user\Desktop\Thesis\Process Pilot project\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/251734 [00:00<?, ?it/s]

{'type': 'file',
 'name': 'BPI_Challenge_2019.xes',
 'path': 'C:\\Users\\user\\Desktop\\Thesis\\Data\\BPI_Challenge_2019.xes',
 'format': 'xes',
 'row_count': 1595923,
 'size_bytes': 728558522}

## Apply the dataset preset mapping

`configs/bpi2019.yaml` was corrected in Phase 1 against this exact file (see
the STATUS note at the top of that file); nothing here is guessed at runtime.

In [3]:
PRESET_PATH = Path("..") / "configs" / "bpi2019.yaml"
mapping, preset = load_preset(PRESET_PATH)
mapping

ColumnMapping(case_id='case:concept:name', activity='concept:name', timestamp='time:timestamp', resource='org:resource', segment_by='case:Item Category', timestamp_format=None, timestamp_timezone_policy='utc')

In [4]:
canonical_df = apply_mapping(raw_df, mapping)
canonical_df.head()

,case_id,activity,timestamp,resource,segment,attr:User,attr:Cumulative net worth (EUR),attr:case:Spend area text,attr:case:Company,attr:case:Document Type,...,attr:case:Purchasing Document,attr:case:Purch. Doc. Category name,attr:case:Vendor,attr:case:Item Type,attr:case:Spend classification text,attr:case:Source,attr:case:Name,attr:case:GR-Based Inv. Verif.,attr:case:Item,attr:case:Goods Receipt
0,2000000000_00001,SRM: Created,2018-01-02 12:53:00,batch_00,"3-way match, invoice before GR",batch_00,298.0,CAPEX & SOCS,companyID_0000,EC Purchase order,...,2000000000,Purchase order,vendorID_0000,Standard,NPR,sourceSystemID_0000,vendor_0000,False,00001,True
1,2000000000_00001,SRM: Complete,2018-01-02 13:53:00,batch_00,"3-way match, invoice before GR",batch_00,298.0,CAPEX & SOCS,companyID_0000,EC Purchase order,...,2000000000,Purchase order,vendorID_0000,Standard,NPR,sourceSystemID_0000,vendor_0000,False,00001,True
2,2000000000_00001,SRM: Awaiting Approval,2018-01-02 13:53:00,batch_00,"3-way match, invoice before GR",batch_00,298.0,CAPEX & SOCS,companyID_0000,EC Purchase order,...,2000000000,Purchase order,vendorID_0000,Standard,NPR,sourceSystemID_0000,vendor_0000,False,00001,True
3,2000000000_00001,SRM: Document Completed,2018-01-02 13:53:00,batch_00,"3-way match, invoice before GR",batch_00,298.0,CAPEX & SOCS,companyID_0000,EC Purchase order,...,2000000000,Purchase order,vendorID_0000,Standard,NPR,sourceSystemID_0000,vendor_0000,False,00001,True
4,2000000000_00001,SRM: In Transfer to Execution Syst.,2018-01-02 13:53:00,batch_00,"3-way match, invoice before GR",batch_00,298.0,CAPEX & SOCS,companyID_0000,EC Purchase order,...,2000000000,Purchase order,vendorID_0000,Standard,NPR,sourceSystemID_0000,vendor_0000,False,00001,True


## Data-quality report

Thresholds come from the preset's `validation` block, not from code. Nothing
is dropped by this step -- `validate()` only reports.

In [5]:
validation_cfg = preset["validation"]
report = validate(
    canonical_df,
    plausible_timestamp_min=validation_cfg["plausible_timestamp_min"],
    plausible_timestamp_max=validation_cfg["plausible_timestamp_max"],
    max_distinct_activities_warn=validation_cfg["max_distinct_activities_warn"],
    min_events_per_case_warn=validation_cfg["min_events_per_case_warn"],
)
pd.set_option("display.max_colwidth", 120)
report.to_frame()

,code,severity,message,count,share,example_rows
0,timestamp_out_of_plausible_range,error,"95 timestamp(s) fall outside the plausible range [2017-01-01 00:00:00, 2020-05-01 00:00:00] (e.g. BPI 2019's junk 19...",95,0.000060,"[{'case_id': '2000000097_00001', 'activity': 'Vendor creates debit memo', 'timestamp': 2001-02-23 22:59:00}, {'case_..."
1,single_event_case,warn,2835 case(s) have fewer than 2 event(s).,2835,0.011262,"[{'case_id': '4507000679_00010', 'activity': 'Create Purchase Order Item', 'timestamp': 2018-01-03 13:04:00}, {'case..."
2,duplicate_event,warn,"209283 row(s) are exact duplicates on ['case_id', 'activity', 'timestamp', 'resource'].",209283,0.131136,"[{'case_id': '2000000022_00001', 'activity': 'Record Invoice Receipt', 'timestamp': 2018-06-29 05:19:00}, {'case_id'..."
3,timestamp_tie_within_case,info,"233463 row(s) share an exact timestamp with another event in the same case, making directly-follows order ambiguous ...",233463,0.146287,"[{'case_id': '2000000000_00001', 'activity': 'SRM: Awaiting Approval', 'timestamp': 2018-01-02 13:53:00}, {'case_id'..."
4,log_size,info,Log has 1595923 event(s) across 251734 case(s).,1595923,1.000000,[]


In [6]:
print(f"events={report.event_count}  cases={report.case_count}  issues={report.issue_count}")

events=1595923  cases=251734  issues=5


## Segment split

`segment_by: case:Item Category` in the preset drives this -- each segment is
a materially different purchase-to-pay flow, run separately from Phase 2 on.

In [7]:
segments = split_by_segment(canonical_df)
{name: {"events": len(df), "cases": df["case_id"].nunique()} for name, df in segments.items()}

{'2-way match': {'events': 5898, 'cases': 1044},
 '3-way match, invoice after GR': {'events': 319233, 'cases': 15182},
 '3-way match, invoice before GR': {'events': 1234708, 'cases': 221010},
 'Consignment': {'events': 36084, 'cases': 14498}}